# Convert Pytorch weight to be tiny-engine compatible

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

In [2]:
# Load model directly
model_name = "Qwen/Qwen3-8B"
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    trust_remote_code=True, 
    torch_dtype=torch.float32,
    device_map="cpu"
)

/root/workspace/qwen_cpu_deployment/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.28it/s]


In [3]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): 

In [4]:
model.model

Qwen3Model(
  (embed_tokens): Embedding(151936, 4096)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
    )
  )
  (norm): Qwen3RMSNorm((

In [5]:
outdir_root = "model_weights/fp32/qwen3_8b"
os.makedirs(outdir_root, exist_ok=True)

## Save lm_head

In [6]:
model.lm_head

Linear(in_features=4096, out_features=151936, bias=False)

In [7]:
output_path = os.path.join(outdir_root, "lm_head.bin")
with torch.no_grad():
    with open(output_path, "wb") as f:
        f.write(model.lm_head._parameters["weight"].cpu().float().numpy().tobytes())
        print(f"lm_head weight saved to: {output_path}")

lm_head weight saved to: model_weights/fp32/qwen3_8b/lm_head.bin


## Save embedding, final norm

In [8]:
embed_tokens = model.model.embed_tokens

In [9]:
final_norm = model.model.norm

In [10]:
OUTPUT_PATH = dict()
output_path = os.path.join(outdir_root, "model")

# export wte
OUTPUT_PATH.update(
    embed_token = ( 
        embed_tokens.weight, os.path.join(output_path, "embed_tokens", "weight.bin") 
    ),
    final_norm = 
    (
        final_norm.weight, os.path.join(output_path, "norm", "weight.bin")
    )
)


for _, path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

# print(OUTPUT_PATH)
with torch.no_grad():
    for name, (data, path) in OUTPUT_PATH.items():
        with open(path, "wb") as f:
            f.write(data.cpu().float().numpy().tobytes())
            print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
            


Saved embed_token to model_weights/fp32/qwen3_8b/model/embed_tokens/weight.bin
Saved final_norm to model_weights/fp32/qwen3_8b/model/norm/weight.bin


## Save Pre-computed rotary embedding with maximum sequence length

In [11]:
rotary_emb = model.model.rotary_emb


In [12]:
rotary_emb.attention_scaling

1.0

In [13]:
rotary_emb.inv_freq


{Tensor:(64,)} tensor([1.0000e+00, 8.0584e-01, 6.4938e-01, 5.2330e-01, 4.2170e-01, 3.3982e-01,
        2.7384e-01, 2.2067e-01, 1.7783e-01, 1.4330e-01, 1.1548e-01, 9.3057e-02,
        7.4989e-02, 6.0430e-02, 4.8697e-02, 3.9242e-02, 3.1623e-02, 2.5483e-02,
        2.0535e-02, 1.6548e-02, 1.3335e-02, 1.0746e-02, 8.6596e-03, 6.9783e-03,
        5.6234e-03, 4.5316e-03, 3.6517e-03, 2.9427e-03, 2.3714e-03, 1.9110e-03,
        1.5399e-03, 1.2409e-03, 1.0000e-03, 8.0584e-04, 6.4938e-04, 5.2330e-04,
        4.2170e-04, 3.3982e-04, 2.7384e-04, 2.2067e-04, 1.7783e-04, 1.4330e-04,
        1.1548e-04, 9.3057e-05, 7.4989e-05, 6.0430e-05, 4.8697e-05, 3.9242e-05,
        3.1623e-05, 2.5483e-05, 2.0535e-05, 1.6548e-05, 1.3335e-05, 1.0746e-05,
        8.6596e-06, 6.9783e-06, 5.6234e-06, 4.5316e-06, 3.6517e-06, 2.9427e-06,
        2.3714e-06, 1.9110e-06, 1.5399e-06, 1.2409e-06])

In [14]:
max_seq_length = 4096
device = model.device
position_ids = torch.arange(max_seq_length, device=device).reshape(1, max_seq_length)
# only provided device info
pseudo_x = torch.randn(1, max_seq_length, 4096, device=device)
cos, sin = rotary_emb(pseudo_x, position_ids)

In [15]:
cos

{Tensor:(1, 4096, 128)} tensor([[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.6925,  0.7965,  ...,  1.0000,  1.0000,  1.0000],
         [-0.4161, -0.0409,  0.2687,  ...,  1.0000,  1.0000,  1.0000],
         ...,
         [-0.8799,  0.9359,  0.9914,  ...,  1.0000,  1.0000,  1.0000],
         [-0.8753,  0.9022,  0.7102,  ...,  1.0000,  1.0000,  1.0000],
         [-0.0660,  0.3139,  0.1399,  ...,  1.0000,  1.0000,  1.0000]]])

In [16]:
OUTPUT_PATH = dict()
# NOTE: All attention layers share the same pre-computed rope rotation matrix
output_path = os.path.join(outdir_root, "model")

# export wte
OUTPUT_PATH.update(
    sin_cache = (sin, os.path.join(output_path, "rotary_emb", "sin_cached.bin") ),
    cos_cache = (cos, os.path.join(output_path, "rotary_emb", "cos_cached.bin") )
)
for _, path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    for name, (data, path) in OUTPUT_PATH.items():
        with open(path, "wb") as f:
            f.write(data.cpu().float().numpy().tobytes())
            print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
    

{'sin_cache': ({Tensor:(1, 4096, 128)} tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 8.4147e-01,  7.2141e-01,  6.0469e-01,  ...,  1.9110e-06,
           1.5399e-06,  1.2409e-06],
         [ 9.0930e-01,  9.9916e-01,  9.6323e-01,  ...,  3.8219e-06,
           3.0799e-06,  2.4819e-06],
         ...,
         [ 4.7523e-01, -3.5230e-01,  1.3118e-01,  ...,  7.8215e-03,
           6.3029e-03,  5.0791e-03],
         [-4.8361e-01,  4.3125e-01,  7.0397e-01,  ...,  7.8234e-03,
           6.3044e-03,  5.0804e-03],
         [-9.9782e-01,  9.4947e-01,  9.9016e-01,  ...,  7.8253e-03,
           6.3060e-03,  5.0816e-03]]]), 'model_weights/fp32/qwen3_8b/model/rotary_emb/sin_cached.bin'), 'cos_cache': ({Tensor:(1, 4096, 128)} tensor([[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
         [ 0.5403,  0.6925,  0.7965,  ...,  1.0000,  1.0000,  1.0000],
         [-0.4161, -0.0409,  0.2687,  ...,  1.0000,  1.0000,  1.0000],

## Save Qwen Decoder Layers

In [17]:
QWEN_BLOCKS = model.model.layers

In [18]:
QWEN_BLOCKS[0]

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
    (act_fn): SiLU()
  )
  (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
)

In [19]:
scaling = QWEN_BLOCKS[0].self_attn.scaling
print(scaling)
scaling = torch.tensor(scaling, dtype=torch.float32)
scaling


0.08838834764831845


{Tensor:()} tensor(0.0884)

In [ ]:
for idx, layer in enumerate(QWEN_BLOCKS):
    OUTPUT_PATH = dict()
    output_path = os.path.join(outdir_root, "model", "layers", f"layer{idx}")
    self_attn = layer.self_attn
    mlp = layer.mlp
    input_layernorm = layer.input_layernorm
    post_attention_layernorm = layer.post_attention_layernorm
    
    # export wte
    OUTPUT_PATH.update(
        # self_attn
        q_proj_weight = ( self_attn.q_proj.weight, os.path.join(output_path, "self_attn", "q_proj", "weight.bin") ),
        k_proj_weight = ( self_attn.k_proj.weight, os.path.join(output_path, "self_attn", "k_proj", "weight.bin") ),
        v_proj_weight = ( self_attn.v_proj.weight, os.path.join(output_path, "self_attn", "v_proj", "weight.bin") ),
        o_proj_weight = ( self_attn.o_proj.weight, os.path.join(output_path, "self_attn", "o_proj", "weight.bin") ),
        q_norm_weight = ( self_attn.q_norm.weight, os.path.join(output_path, "self_attn", "q_norm", "weight.bin") ),
        k_norm_weight = ( self_attn.k_norm.weight, os.path.join(output_path, "self_attn", "k_norm", "weight.bin") ),
        # scaling factor in attention calculation
        scaling = (torch.tensor(self_attn.scaling, dtype=torch.float32), os.path.join(output_path, "self_attn", "scaling.bin") ),
        # mlp
        gate_proj_weight = (mlp.gate_proj.weight, os.path.join(output_path, "mlp", "gate_proj", "weight.bin") ),
        up_proj_weight   = (mlp.up_proj.weight, os.path.join(output_path, "mlp", "up_proj", "weight.bin") ),
        down_proj_weight = (mlp.down_proj.weight, os.path.join(output_path, "mlp", "down_proj", "weight.bin") ),
        # input_layernorm
        input_layernorm_weight = (input_layernorm.weight, os.path.join(output_path, "input_layernorm", "weight.bin") ),
        # post_attention_layernorm
        post_attention_layernorm_weight = (post_attention_layernorm.weight, os.path.join(output_path, "post_attention_layernorm", "weight.bin") ),

    )

    for _, path in OUTPUT_PATH.values():
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)

    with torch.no_grad():
        for name, (data, path) in OUTPUT_PATH.items():
            with open(path, "wb") as f:
                f.write(data.cpu().float().numpy().tobytes())
                print(f"\033[92mSaved {name} to {path}\033[0m")  # Green colored output
    

Saved q_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/q_proj/weight.bin
Saved k_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/k_proj/weight.bin
Saved v_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/v_proj/weight.bin
Saved o_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/o_proj/weight.bin
Saved q_norm_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/q_norm/weight.bin
Saved k_norm_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/k_norm/weight.bin
Saved scaling to model_weights/fp32/qwen3_8b/model/layers/layer0/self_attn/scaling.bin
Saved gate_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/mlp/gate_proj/weight.bin
Saved up_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/mlp/up_proj/weight.bin
Saved down_proj_weight to model_weights/fp32/qwen3_8b/model/layers/layer0/mlp/down_proj/weight.bin
Saved input_layernorm_weig

: 

# Dump activation and input to validate cpp implementation



## Attention module

### register input and output hook

In [1]:
inputs_dict = []
outputs_dict = []
def hook_func(module, input, output):
    # print("=module=")
    # print(module)
    # print("=input=")
    # print(input)
    # print("=output=")
    # print(output)
    inputs_dict.append(input)
    outputs_dict.append(output)

hook = model.transformer.h[0].attn.register_forward_hook(hook_func)
hook = model.transformer.h[0].attn.register_forward_hook(hook_func)

NameError: name 'model' is not defined

In [9]:
# hook.remove()

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

In [2]:
# Load model directly
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True, fp32=True).eval()

Your device support faster inference by passing bf16=True in "AutoModelForCausalLM.from_pretrained".


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True)

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation. But if you use transformers>=4.32.0, there is no need to do this.
# model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True) # 可指定不同的生成长度、top_p等相关超参

# 第一轮对话 1st dialogue turn
response, history = model.chat(tokenizer, "你好", history=None)
print(response)

/root/miniconda3/envs/tinyml/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1614: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [5]:
input = inputs_dict[0][0]
output_attn = outputs_dict[0][0]
input, output_attn

({Tensor:(1, 20, 4096)} tensor([[[ 0.0099, -0.0241,  0.0344,  ..., -0.0008, -0.0185, -0.0072],
          [-0.1132, -0.1359, -0.0490,  ..., -0.0123,  0.0712,  0.0126],
          [-0.0245,  0.0012, -0.0050,  ...,  0.0014,  0.0277, -0.0156],
          ...,
          [ 0.0099, -0.0241,  0.0344,  ..., -0.0008, -0.0185, -0.0072],
          [-0.0664, -0.1269, -0.0096,  ..., -0.0835, -0.0816, -0.1260],
          [-0.0245,  0.0012, -0.0050,  ...,  0.0014,  0.0277, -0.0156]]]),
 {Tensor:(1, 20, 4096)} tensor([[[-0.0776,  0.0330, -0.0400,  ..., -0.0069, -0.0060, -0.0161],
          [-0.0454, -0.0116, -0.1370,  ..., -0.0124,  0.0286,  0.0095],
          [-0.0014, -0.0037, -0.0754,  ..., -0.0416,  0.0301, -0.0044],
          ...,
          [-0.0511, -0.0104, -0.0009,  ...,  0.0211,  0.0202, -0.0037],
          [-0.0931,  0.1034, -0.0824,  ..., -0.0035,  0.0635,  0.0278],
          [-0.0518,  0.0270, -0.0484,  ...,  0.0714,  0.0171,  0.0130]]]))

In [6]:
import math
import struct

OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer", f"layer0")

# export wte
OUTPUT_PATH.update(
    input_attn = os.path.join(output_path, "self_attn/activation/input_attn.bin"),
    output_attn = os.path.join(output_path, "self_attn/activation/output_attn.bin"),
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['input_attn'], "wb") as f:
        f.write(input.cpu().numpy().tobytes())
    with open(OUTPUT_PATH['output_attn'], "wb") as f:
        f.write(output_attn.cpu().numpy().tobytes())

{'input_attn': 'qwen-7b-chat/transformer/layer0/self_attn/activation/input_attn.bin', 'output_attn': 'qwen-7b-chat/transformer/layer0/self_attn/activation/output_attn.bin'}


In [20]:
len(inputs_dict)

9

## QwenBlock

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True, fp32=True).eval()

/root/miniconda3/envs/tinyml/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1614: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Your device support faster inference by passing bf16=True in "AutoModelForCausalLM.from_pretrained".


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [2]:

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation. But if you use transformers>=4.32.0, there is no need to do this.
# model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True) # 可指定不同的生成长度、top_p等相关超参

# hook.remove()
inputs_dict_qwen_block = []
outputs_dict_qwen_block = []
def hook_func(module, input, output):
    # print("=module=")
    # print(module)
    # print("=input=")
    # print(input)
    # print("=output=")
    # print(output)
    inputs_dict_qwen_block.append(input)
    outputs_dict_qwen_block.append(output)

hook = model.transformer.h[0].register_forward_hook(hook_func)



# 第一轮对话 1st dialogue turn
response, history = model.chat(tokenizer, "你好", history=None)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


你好！有什么我能帮助你的吗？


In [5]:
input_block = inputs_dict_qwen_block[0][0]
output_block = outputs_dict_qwen_block[0][0]
input_block, output_block


({Tensor:(1, 20, 4096)} tensor([[[ 1.0490e-04, -2.8229e-04,  3.3951e-04,  ..., -8.7023e-06,
           -2.0027e-04, -7.9155e-05],
          [-1.9409e-02, -2.5757e-02, -7.8125e-03,  ..., -2.1973e-03,
            1.2451e-02,  2.2430e-03],
          [-4.4250e-03,  2.3651e-04, -8.3542e-04,  ...,  2.6894e-04,
            5.0964e-03, -2.9297e-03],
          ...,
          [ 1.0490e-04, -2.8229e-04,  3.3951e-04,  ..., -8.7023e-06,
           -2.0027e-04, -7.9155e-05],
          [-9.8877e-03, -2.0874e-02, -1.3351e-03,  ..., -1.2939e-02,
           -1.2390e-02, -1.9531e-02],
          [-4.4250e-03,  2.3651e-04, -8.3542e-04,  ...,  2.6894e-04,
            5.0964e-03, -2.9297e-03]]]),
 {Tensor:(1, 20, 4096)} tensor([[[-2.2915e-01,  1.4635e-01, -1.3796e-01,  ..., -1.1090e-01,
            3.9673e-03, -8.9641e-02],
          [-8.8610e-02, -1.9821e-02, -4.1045e-02,  ..., -4.1199e-02,
            6.7500e-02,  6.0080e-02],
          [ 9.2342e-02, -1.4332e-01, -1.1347e-01,  ..., -1.5313e-01,
           

In [6]:
import math
import struct

OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer", f"layer0")

# export wte
OUTPUT_PATH.update(
    input_block = os.path.join(output_path, "activation/input.bin"),
    output_block = os.path.join(output_path, "activation/output.bin"),
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['input_block'], "wb") as f:
        f.write(input_block.cpu().numpy().tobytes())
    with open(OUTPUT_PATH['output_block'], "wb") as f:
        f.write(output_block.cpu().numpy().tobytes())

{'input_block': 'qwen-7b-chat/transformer/layer0/activation/input.bin', 'output_block': 'qwen-7b-chat/transformer/layer0/activation/output.bin'}
